In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [ ]:
# Load the volume data
patho = pd.read_csv("../Dataset002_DFGFinetuned_2d_full_pathologic___volumes.csv")

In [ ]:
# Load the metadata
patho_metadata = pd.read_excel("../MSData_noNMO.xlsx")

In [ ]:
# Adjust the column name for the patient ID (replace 'actual_patient_id_column' with the correct name)
metadata_id_column = "patID"  # Replace this with the actual column name in the metadata
patho_id_column = "subject_id"  # Replace this with the column name in the volumes DataFrame (patho)

In [ ]:
# Merge the volume data with metadata
patho_metadata = patho_metadata.merge(patho[['subject_id', 'volume']], left_on=metadata_id_column, right_on=patho_id_column)

In [ ]:
# Filter for specific disease types (if applicable)
disease_types = patho_metadata['Disease Course/ Type at MRI date'].unique()
print(f"Available Disease Types: {disease_types}")

In [ ]:
# Scatter plot of volume vs. disease duration for each disease type
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=patho_metadata,
    x='DisDur_Months',
    y='volume',
    hue='Disease Course/ Type at MRI date',
    alpha=0.7
)
plt.title("Volume vs. Disease Duration by Disease Type")
plt.xlabel("Disease Duration (Months)")
plt.ylabel("Volume")
plt.legend(title="Disease Type")
plt.savefig("volume_vs_disease_duration.png")
plt.show()

In [ ]:
# Fit a linear regression model with interaction terms
model = smf.ols(
    formula="volume ~ DisDur_Months * Q('Disease Course/ Type at MRI date')",
    data=patho_metadata
).fit()

print(model.summary())

In [ ]:
# Pairwise comparisons
# If the interaction is significant, test the slopes for each disease type individually
grouped = patho_metadata.groupby('Disease Course/ Type at MRI date')

for name, group in grouped:
    print(f"\nDisease Type: {name}")
    single_model = smf.ols("volume ~ DisDur_Months", data=group).fit()
    print(single_model.summary())